# Model Development and Evaluation
> **Project:** AI-Based Product Demand Forecasting System  
> **Objective:** Train and evaluate multiple ML models (Linear Regression, Random Forest, XGBoost, LightGBM, CatBoost) on the engineered feature set and identify the best performer.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import joblib
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (14, 5)
sns.set_style('whitegrid')

FEATURES_PATH = '../notebook/data/features_data.csv'
MODELS_DIR    = '../models/'
os.makedirs(MODELS_DIR, exist_ok=True)
print('Libraries loaded.')

In [ ]:
# ── Load feature data ───────────────────────────────────────────
df = pd.read_csv(FEATURES_PATH, parse_dates=['Date'])
df = df.sort_values('Date').reset_index(drop=True)

TARGET = 'TotalQuantity'
DROP_COLS = [TARGET, 'Date', 'TotalRevenue']
FEATURE_COLS = [c for c in df.columns if c not in DROP_COLS]

X = df[FEATURE_COLS].fillna(0)
y = df[TARGET]

print(f'Features: {len(FEATURE_COLS)}, Samples: {len(X)}')
print('Feature list:', FEATURE_COLS)

In [ ]:
# ── Temporal train/test split (80 / 20) ─────────────────────────
split_idx = int(len(X) * 0.80)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
dates_test = df['Date'].iloc[split_idx:]

print(f'Train size: {len(X_train):,}  ({df["Date"].iloc[0].date()} → {df["Date"].iloc[split_idx-1].date()})')
print(f'Test  size: {len(X_test):,}  ({df["Date"].iloc[split_idx].date()} → {df["Date"].iloc[-1].date()})')

In [ ]:
# ── Scale features (required for Linear Regression) ────────────
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

joblib.dump(scaler, os.path.join(MODELS_DIR, 'scaler.pkl'))
print('Scaler fitted and saved.')

In [ ]:
# ── Helper: evaluation metrics ──────────────────────────────────
def mape(y_true, y_pred):
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mp   = mape(np.array(y_true), np.array(y_pred))
    print(f'{name:<20} MAE={mae:8.1f}  RMSE={rmse:8.1f}  R²={r2:.4f}  MAPE={mp:.2f}%')
    return {'Model': name, 'MAE': round(mae,2), 'RMSE': round(rmse,2),
            'R2': round(r2,4), 'MAPE': round(mp,2)}

results = []

In [ ]:
# ── Model 1: Linear Regression ──────────────────────────────────
lr = LinearRegression()
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)

results.append(evaluate('Linear Regression', y_test, y_pred_lr))
joblib.dump(lr, os.path.join(MODELS_DIR, 'linear_regression.pkl'))

In [ ]:
# ── Model 2: Random Forest ──────────────────────────────────────
rf = RandomForestRegressor(
    n_estimators=200, max_depth=10, min_samples_leaf=5,
    n_jobs=-1, random_state=42
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

results.append(evaluate('Random Forest', y_test, y_pred_rf))
joblib.dump(rf, os.path.join(MODELS_DIR, 'random_forest.pkl'))

In [ ]:
# ── Model 3: XGBoost ────────────────────────────────────────────
xgb = XGBRegressor(
    n_estimators=300, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, verbosity=0
)
xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)],
        verbose=False)
y_pred_xgb = xgb.predict(X_test)

results.append(evaluate('XGBoost', y_test, y_pred_xgb))
joblib.dump(xgb, os.path.join(MODELS_DIR, 'xgboost.pkl'))

In [ ]:
# ── Model 4: LightGBM ───────────────────────────────────────────
lgbm = LGBMRegressor(
    n_estimators=300, learning_rate=0.05, max_depth=6,
    num_leaves=63, subsample=0.8, colsample_bytree=0.8,
    random_state=42, verbose=-1
)
lgbm.fit(X_train, y_train)
y_pred_lgbm = lgbm.predict(X_test)

results.append(evaluate('LightGBM', y_test, y_pred_lgbm))
joblib.dump(lgbm, os.path.join(MODELS_DIR, 'lightgbm.pkl'))

In [ ]:
# ── Model 5: CatBoost ───────────────────────────────────────────
cat = CatBoostRegressor(
    iterations=300, learning_rate=0.05, depth=6,
    random_seed=42, verbose=0
)
cat.fit(X_train, y_train)
y_pred_cat = cat.predict(X_test)

results.append(evaluate('CatBoost', y_test, y_pred_cat))
joblib.dump(cat, os.path.join(MODELS_DIR, 'catboost.pkl'))

In [ ]:
# ── Model comparison table ──────────────────────────────────────
results_df = pd.DataFrame(results).set_index('Model')
results_df = results_df.sort_values('RMSE')
print('\n=== Model Comparison ===')
print(results_df.to_string())

# Save results
results_df.to_csv('../notebook/data/model_results.csv')

In [ ]:
# ── Predictions vs Actual ────────────────────────────────────────
best_name = results_df.index[0]
pred_map = {
    'Linear Regression': y_pred_lr,
    'Random Forest':     y_pred_rf,
    'XGBoost':           y_pred_xgb,
    'LightGBM':          y_pred_lgbm,
    'CatBoost':          y_pred_cat,
}

plt.figure(figsize=(14, 5))
plt.plot(dates_test.values, y_test.values, label='Actual', linewidth=1.5, color='black')
plt.plot(dates_test.values, pred_map[best_name], label=f'{best_name} (best)',
         linewidth=1.5, linestyle='--', color='crimson')
plt.title(f'Actual vs {best_name} Predictions – Test Period')
plt.xlabel('Date')
plt.ylabel('Daily Quantity')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Residual analysis ────────────────────────────────────────────
best_preds = pred_map[best_name]
residuals  = np.array(y_test) - np.array(best_preds)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(best_preds, residuals, alpha=0.4, s=10, color='darkorange')
axes[0].axhline(0, color='black', linewidth=1)
axes[0].set_title('Residuals vs Predicted')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Residual')

axes[1].hist(residuals, bins=40, color='steelblue', edgecolor='white')
axes[1].set_title('Residual Distribution')
axes[1].set_xlabel('Residual')

plt.tight_layout()
plt.show()

In [ ]:
# ── Feature importance – best tree model ─────────────────────────
tree_models = {'Random Forest': rf, 'XGBoost': xgb, 'LightGBM': lgbm, 'CatBoost': cat}

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for idx, (mname, model) in enumerate(tree_models.items()):
    imp = pd.Series(model.feature_importances_, index=FEATURE_COLS)
    imp.sort_values().tail(10).plot(kind='barh', ax=axes[idx],
                                     color='teal', edgecolor='black')
    axes[idx].set_title(f'Feature Importance – {mname}')
    axes[idx].set_xlabel('Importance')

plt.suptitle('Top-10 Feature Importances per Model', y=1.01)
plt.tight_layout()
plt.show()

## Model Development Conclusions

| Model | Strengths | Weaknesses |
|-------|-----------|------------|
| Linear Regression | Fast, interpretable | Assumes linearity; weak on patterns |
| Random Forest | Robust, handles non-linearity | Slow on large data; memory heavy |
| XGBoost | High accuracy, regularisation | Requires tuning; slower than LGBM |
| LightGBM | Fastest tree model | Can overfit with few data points |
| CatBoost | Best with categoricals, no tuning | Slow training by default |

> **Key finding:** Gradient-boosted tree models (XGBoost / LightGBM / CatBoost) consistently outperform Linear Regression on this non-linear demand signal.  
> **Next step:** Combine model predictions in an ensemble (Notebook 05).